
# Bulk RNA-seq End-to-End Pipeline (Mouse GRCm38.84)

This notebook performs:

1. FASTQ detection and samplesheet generation
2. nf-core/rnaseq execution using `%%bash`
3. Formatting count matrices
4. Single-cell reference loading
5. Cell-type deconvolution using NNLS

Reference genome/library:
- Mouse: `GRCm38.84`


In [1]:
import os
os.environ['PATH'] = '/home/nakagawa/anaconda3/envs/scatac/bin:' + os.environ['PATH']
print(os.environ['PATH'].split(':')[0])  # should print the scatac bin path

/home/nakagawa/anaconda3/envs/scatac/bin


In [2]:

# Paths
BASE_DIR = "/home/nakagawa/datasets/SRR_spinalcord"
RNA_DIR = f"{BASE_DIR}/bulkRNA"

RESULTS_DIR = f"{BASE_DIR}/results_rna"
SAMPLESHEET = f"{BASE_DIR}/rna_samplesheet.csv"

H5AD_REF = "/home/nakagawa/datasets/h5ad/10X_cells_v3_AIBS.h5ad"


In [3]:

import os
import glob
import pandas as pd

def create_nfcore_samplesheet(fastq_dir, output_csv):
    r1_files = sorted(glob.glob(os.path.join(fastq_dir, "*_1.fastq.gz")))

    records = []

    for r1 in r1_files:
        sample = os.path.basename(r1).split("_1.fastq.gz")[0]
        r2 = os.path.join(fastq_dir, f"{sample}_2.fastq.gz")

        if os.path.exists(r2):
            records.append([sample, r1, r2, "auto"])
        else:
            print(f"Missing R2 for {sample}")

    df = pd.DataFrame(
        records,
        columns=["sample", "fastq_1", "fastq_2", "strandedness"]
    )

    df.to_csv(output_csv, index=False)

    print(f"Saved samplesheet to: {output_csv}")
    print(df.head())

    return df

rna_sheet = create_nfcore_samplesheet(RNA_DIR, SAMPLESHEET)


Saved samplesheet to: /home/nakagawa/datasets/SRR_spinalcord/rna_samplesheet.csv
        sample                                            fastq_1  \
0  SRR18338966  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...   
1  SRR18338967  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...   
2  SRR18338968  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...   
3  SRR18338969  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...   
4  SRR18338970  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...   

                                             fastq_2 strandedness  
0  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...         auto  
1  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...         auto  
2  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...         auto  
3  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...         auto  
4  /home/nakagawa/datasets/SRR_spinalcord/bulkRNA...         auto  



## Run nf-core/rnaseq

This cell uses `%%bash` so it can be executed directly inside Jupyter.

Recommended:
- Run notebook inside `tmux`
- Use `-resume` to recover interrupted runs


In [4]:
%%bash

cd /home/nakagawa/datasets/SRR_spinalcord

nextflow run nf-core/rnaseq \
    -profile singularity \
    --input rna_samplesheet.csv \
    --outdir ./results_rna \
    --genome GRCm38 \
    --aligner star_salmon \
    -resume


nloading nextflow dependencies. It may require a few seconds, please wait .. 

CAPSULE: Downloading dependency org.multiverse:multiverse-core:pom:0.7.0
CAPSULE: Transfer failed: capsule.org.eclipse.aether.transfer.ArtifactTransferException: Could not transfer artifact org.multiverse:multiverse-core:pom:0.7.0 from/to central (https://repo1.maven.org/maven2/): Received fatal alert: handshake_failure (for stack trace, run with -Dcapsule.log=verbose)
CAPSULE: Downloading dependency org.iq80.leveldb:leveldb-api:pom:0.7
CAPSULE: Transfer failed: capsule.org.eclipse.aether.transfer.ArtifactTransferException: Could not transfer artifact org.iq80.leveldb:leveldb-api:pom:0.7 from/to central (https://repo1.maven.org/maven2/): Received fatal alert: handshake_failure (for stack trace, run with -Dcapsule.log=verbose)
CAPSULE: Downloading dependency io.nextflow:nxf-commons:pom:0.24.2
CAPSULE: Transfer failed: capsule.org.eclipse.aether.transfer.ArtifactTransferException: Could not transfer artifact io.nextflow:nxf-commons:pom:0.24.2 from/to central (https://repo1.maven.org/mave

Unable to initialize nextflow environment


In [ ]:

import os
import pandas as pd

rna_counts_path = os.path.join(
    RESULTS_DIR,
    "star_salmon",
    "salmon.merged.gene_counts.tsv"
)

print("Loading RNA counts...")

rna_df = pd.read_csv(rna_counts_path, sep="\t")

rna_df = rna_df.drop(columns=["gene_id"])

rna_df = rna_df.groupby("gene_name").sum()

output_csv = os.path.join(BASE_DIR, "bulkRNA_counts.csv")

rna_df.to_csv(output_csv)

print(f"Saved formatted counts to: {output_csv}")

rna_df.head()


In [ ]:

import anndata as ad
import scanpy as sc
import numpy as np
from scipy.optimize import nnls
import matplotlib.pyplot as plt

bulk_rna = pd.read_csv(
    f"{BASE_DIR}/bulkRNA_counts.csv",
    index_col=0
)

adata_ref = sc.read_h5ad(H5AD_REF)

print(bulk_rna.shape)
print(adata_ref.shape)


In [ ]:

sc.pp.normalize_total(adata_ref, target_sum=1e4)
sc.pp.log1p(adata_ref)

# Adjust if your annotation column differs
cell_type_key = "cell_type"

sc.tl.rank_genes_groups(
    adata_ref,
    groupby=cell_type_key,
    method="wilcoxon"
)

top_n_markers = 100

markers = []

for cl in adata_ref.obs[cell_type_key].unique():
    genes = sc.get.rank_genes_groups_df(
        adata_ref,
        group=cl
    )["names"].head(top_n_markers)

    markers.extend(genes)

overlapping = list(
    set(markers)
    & set(bulk_rna.index)
    & set(adata_ref.var_names)
)

print(f"Overlapping genes: {len(overlapping)}")


In [ ]:

signature_matrix = pd.DataFrame(index=overlapping)

for cl in adata_ref.obs[cell_type_key].unique():
    subset = adata_ref[adata_ref.obs[cell_type_key] == cl]

    mean_expr = np.ravel(
        subset[:, overlapping].X.mean(axis=0)
    )

    signature_matrix[cl] = mean_expr

signature_matrix.head()


In [ ]:

deconv = pd.DataFrame(
    index=bulk_rna.columns,
    columns=signature_matrix.columns
)

A = signature_matrix.loc[overlapping].values

for sample in bulk_rna.columns:
    b = bulk_rna.loc[overlapping, sample].values

    sol, _ = nnls(A, b)

    if sol.sum() > 0:
        sol = sol / sol.sum()

    deconv.loc[sample] = sol

deconv = deconv.astype(float)

deconv


In [ ]:

deconv.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)

plt.title("Bulk RNA-seq Deconvolution")
plt.ylabel("Cell Fraction")
plt.xlabel("Samples")

plt.tight_layout()
plt.show()
